In [1]:
!pip install transformers

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0 -> 26.1.2
[notice] To update, run: C:\Users\liano\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [14]:
import requests
import json

ticker = 'SBER'

url = f"https://iss.moex.com/iss/securities/{ticker}.json"
response = requests.get(url).json()

description_block = response['description']

columns = description_block['columns']
data_rows = description_block['data']

name_idx = columns.index('name')
title_idx = columns.index('title')
value_idx = columns.index('value')

print(f"=== Реальное описание тикера {ticker} из MOEX API ===\n")
print(f"{'Код поля':<12} | {'Название параметра':<35} | {'Значение в API'}")
print("-" * 75)

for row in data_rows:
    field_code = row[name_idx]
    field_title = row[title_idx]
    field_value = row[value_idx]
    
    print(f"{field_code:<12} | {field_title:<35} | {field_value}")

=== Реальное описание тикера SBER из MOEX API ===

Код поля     | Название параметра                  | Значение в API
---------------------------------------------------------------------------
SECID        | Код ценной бумаги                   | SBER
ISSUENAME    | Наименование ценной бумаги          | Акции обыкновенные
NAME         | Полное наименование                 | Сбербанк России ПАО ао
SHORTNAME    | Краткое наименование                | Сбербанк
ISIN         | ISIN код                            | RU0009029540
REGNUMBER    | Номер государственной регистрации   | 10301481B
ISSUESIZE    | Объем выпуска                       | 21586948000
FACEVALUE    | Номинальная стоимость               | 3
FACEUNIT     | Валюта номинала                     | SUR
ISSUEDATE    | Дата начала торгов                  | 2007-07-20
LATNAME      | Английское наименование             | Sberbank
HASPROSPECTUS | Наличие проспекта                   | 0
HASDEFAULT   | Допущен дефолт                    

In [16]:
import random
from datetime import datetime, timedelta
import pandas as pd
import numpy as np

random.seed(42)
np.random.seed(42)

# =====================================================================
# 1. СТРУКТУРИРОВАНИЕ РЕАЛЬНЫХ ТИКЕРОВ И СОЗДАНИЕ ТАБЛИЦЫ ДОМЕНОВ
# =====================================================================

ticker_to_sector = {
    # НЕФТЬ И ГАЗ (35 тикеров)
    'GAZP': 'НЕФТЬ И ГАЗ', 'LKOH': 'НЕФТЬ И ГАЗ', 'NVTK': 'НЕФТЬ И ГАЗ', 'TATN': 'НЕФТЬ И ГАЗ', 
    'ROSN': 'НЕФТЬ И ГАЗ', 'SNGS': 'НЕФТЬ И ГАЗ', 'BANEP': 'НЕФТЬ И ГАЗ', 'SNGSP': 'НЕФТЬ И ГАЗ',
    'TATNP': 'НЕФТЬ И ГАЗ', 'BANE': 'НЕФТЬ И ГАЗ', 'RNFT': 'НЕФТЬ И ГАЗ', 'GAZPP': 'НЕФТЬ И ГАЗ',
    'SNGNP': 'НЕФТЬ И ГАЗ', 'NVTKP': 'НЕФТЬ И ГАЗ', 'ROSNP': 'НЕФТЬ И ГАЗ', 'YAKG': 'НЕФТЬ И ГАЗ',
    'IRKT': 'НЕФТЬ И ГАЗ', 'ZNTO': 'НЕФТЬ И ГАЗ', 'KMZN': 'НЕФТЬ И ГАЗ', 'REFT': 'НЕФТЬ И ГАЗ',
    'TOMG': 'НЕФТЬ И ГАЗ', 'SIBNE': 'НЕФТЬ И ГАЗ', 'ASTN': 'НЕФТЬ И ГАЗ', 'SNGN': 'НЕФТЬ И ГАЗ',
    'TATNG': 'НЕФТЬ И ГАЗ', 'GAZMS': 'НЕФТЬ И ГАЗ', 'LKOHP': 'НЕФТЬ И ГАЗ', 'UFAH': 'НЕФТЬ И ГАЗ',
    'NGRK': 'НЕФТЬ И ГАЗ', 'PSNG': 'НЕФТЬ И ГАЗ', 'PECH': 'НЕФТЬ И ГАЗ', 'YAMG': 'НЕФТЬ И ГАЗ',
    'SRENG': 'НЕФТЬ И ГАЗ', 'OCNG': 'НЕФТЬ И ГАЗ', 'SFTN': 'НЕФТЬ И ГАЗ',

    # ФИНАНСЫ (35 тикеров)
    'BSPB': 'ФИНАНСЫ', 'CBOM': 'ФИНАНСЫ', 'DOMRF': 'ФИНАНСЫ', 'MOEX': 'ФИНАНСЫ', 'SBER': 'ФИНАНСЫ', 
    'VTBR': 'ФИНАНСЫ', 'SBERP': 'ФИНАНСЫ', 'TCSG': 'ФИНАНСЫ', 'WUSH': 'ФИНАНСЫ', 'QIWI': 'ФИНАНСЫ',
    'SPBE': 'ФИНАНСЫ', 'RENI': 'ФИНАНСЫ', 'HNFG': 'ФИНАНСЫ', 'CARM': 'ФИНАНСЫ', 'LEAS': 'ФИНАНСЫ',
    'AKBB': 'ФИНАНСЫ', 'ALFB': 'ФИНАНСЫ', 'ABNK': 'ФИНАНСЫ', 'ZABK': 'ФИНАНСЫ', 'UBNK': 'ФИНАНСЫ',
    'MBNK': 'ФИНАНСЫ', 'EBNK': 'ФИНАНСЫ', 'TCSGP': 'ФИНАНСЫ', 'BSFNB': 'ФИНАНСЫ', 'VTBRP': 'ФИНАНСЫ',
    'SPBFP': 'ФИНАНСЫ', 'ZMKG': 'ФИНАНСЫ', 'FINM': 'ФИНАНСЫ', 'STLN': 'ФИНАНСЫ', 'KRED': 'ФИНАНСЫ',
    'MTRK': 'ФИНАНСЫ', 'STRG': 'ФИНАНСЫ', 'PRMB': 'ФИНАНСЫ', 'INVB': 'ФИНАНСЫ', 'CCBM': 'ФИНАНСЫ',

    # МЕТАЛЛУРГИЯ (35 тикеров)
    'CHMF': 'МЕТАЛЛУРГИЯ', 'GMKN': 'МЕТАЛЛУРГИЯ', 'MAGN': 'МЕТАЛЛУРГИЯ', 'PLZL': 'МЕТАЛЛУРГИЯ', 
    'POLY': 'МЕТАЛЛУРГИЯ', 'SELG': 'МЕТАЛЛУРГИЯ', 'NLMK': 'МЕТАЛЛУРГИЯ', 'RUAL': 'МЕТАЛЛУРГИЯ', 
    'MMK': 'МЕТАЛЛУРГИЯ', 'VSMO': 'МЕТАЛЛУРГИЯ', 'KAZT': 'МЕТАЛЛУРГИЯ', 'MSTT': 'МЕТАЛЛУРГИЯ',
    'CHMFP': 'МЕТАЛЛУРГИЯ', 'NLMKP': 'МЕТАЛЛУРГИЯ', 'PLZLP': 'МЕТАЛЛУРГИЯ', 'SELGP': 'МЕТАЛЛУРГИЯ',
    'RUALP': 'МЕТАЛЛУРГИЯ', 'NMTL': 'МЕТАЛЛУРГИЯ', 'STML': 'МЕТАЛЛУРГИЯ', 'KMTL': 'МЕТАЛЛУРГИЯ',
    'UMTL': 'МЕТАЛЛУРГИЯ', 'AMTL': 'МЕТАЛЛУРГИЯ', 'SSTL': 'МЕТАЛЛУРГИЯ', 'GZML': 'МЕТАЛЛУРГИЯ',
    'BRZL': 'МЕТАЛЛУРГИЯ', 'CHZN': 'МЕТАЛЛУРГИЯ', 'NKML': 'МЕТАЛЛУРГИЯ', 'VTML': 'МЕТАЛЛУРГИЯ',
    'NKMK': 'МЕТАЛЛУРГИЯ', 'TMK': 'МЕТАЛЛУРГИЯ', 'TMKP': 'МЕТАЛЛУРГИЯ', 'UZMZ': 'МЕТАЛЛУРГИЯ',
    'SKMZ': 'МЕТАЛЛУРГИЯ', 'KMZG': 'МЕТАЛЛУРГИЯ', 'MCHM': 'МЕТАЛЛУРГИЯ',

    # ЭНЕРГЕТИКА (35 тикеров)
    'IRAO': 'ЭНЕРГЕТИКА', 'MSNG': 'ЭНЕРГЕТИКА', 'ENPG': 'ЭНЕРГЕТИКА', 'HYDR': 'ЭНЕРГЕТИКА', 
    'UPRO': 'ЭНЕРГЕТИКА', 'FEES': 'ЭНЕРГЕТИКА', 'TGKA': 'ЭНЕРГЕТИКА', 'OGKB': 'ЭНЕРГЕТИКА', 
    'TGKN': 'ЭНЕРГЕТИКА', 'MRKS': 'ЭНЕРГЕТИКА', 'MRKP': 'ЭНЕРГЕТИКА', 'KUBE': 'ЭНЕРГЕТИКА',
    'TGKDP': 'ЭНЕРГЕТИКА', 'TGKB': 'ЭНЕРГЕТИКА', 'TGKC': 'ЭНЕРГЕТИКА', 'TGKE': 'ЭНЕРГЕТИКА',
    'MRKV': 'ЭНЕРГЕТИКА', 'MRKU': 'ЭНЕРГЕТИКА', 'MRKZ': 'ЭНЕРГЕТИКА', 'MRKC': 'ЭНЕРГЕТИКА',
    'MRKK': 'ЭНЕРГЕТИКА', 'MSRS': 'ЭНЕРГЕТИКА', 'TASB': 'ЭНЕРГЕТИКА', 'YAKG_E': 'ЭНЕРГЕТИКА',
    'VLOG': 'ЭНЕРГЕТИКА', 'SBER_E': 'ЭНЕРГЕТИКА', 'LSNG': 'ЭНЕРГЕТИКА', 'LSNGP': 'ЭНЕРГЕТИКА',
    'KROT': 'ЭНЕРГЕТИКА', 'ELTZ': 'ЭНЕРГЕТИКА', 'ESTG': 'ЭНЕРГЕТИКА', 'WTGC': 'ЭНЕРГЕТИКА',
    'NENG': 'ЭНЕРГЕТИКА', 'KENG': 'ЭНЕРГЕТИКА', 'DVEC': 'ЭНЕРГЕТИКА',

    # ИТ (30 тикеров)
    'HEAD': 'ИТ', 'CNRU': 'ИТ', 'YNDX': 'ИТ', 'OZON': 'ИТ', 'POSI': 'ИТ', 'VKCO': 'ИТ', 
    'DIOD': 'ИТ', 'ASTR': 'ИТ', 'SOFL': 'ИТ', 'WAVE': 'ИТ', 'IVNF': 'ИТ', 'SFTC': 'ИТ',
    'CLOD': 'ИТ', 'DATA': 'ИТ', 'CYBR': 'ИТ', 'KASP': 'ИТ', 'ERTC': 'ИТ', 'ELMT': 'ИТ',
    'RESO': 'ИТ', 'FIND': 'ИТ', 'AI_T': 'ИТ', 'BCKL': 'ИТ', 'SFTG': 'ИТ', 'TEHN': 'ИТ',
    'MIND': 'ИТ', 'GLON': 'ИТ', 'NEO_I': 'ИТ', 'ITCO': 'ИТ', 'KODX': 'ИТ', 'LOGI': 'ИТ',

    # ТРАНСПОРТ (25 тикеров)
    'AFLT': 'ТРАНСПОРТ', 'FLOT': 'ТРАНСПОРТ', 'NMTP': 'ТРАНСПОРТ', 'GLTR': 'ТРАНСПОРТ', 
    'UTAR': 'ТРАНСПОРТ', 'FESH': 'ТРАНСПОРТ', 'NKHP': 'ТРАНСПОРТ', 'GTRK': 'ТРАНСПОРТ',
    'DELI': 'ТРАНСПОРТ', 'AFLTP': 'ТРАНСПОРТ', 'FLOTP': 'ТРАНСПОРТ', 'SAPT': 'ТРАНСПОРТ',
    'TRFT': 'ТРАНСПОРТ', 'RZD_T': 'ТРАНСПОРТ', 'AMTP': 'ТРАНСПОРТ', 'VLOG_T': 'ТРАНСПОРТ',
    'KTRA': 'ТРАНСПОРТ', 'PTRA': 'ТРАНСПОРТ', 'LOGS': 'ТРАНСПОРТ', 'SHP_T': 'ТРАНСПОРТ',
    'AVTO': 'ТРАНСПОРТ', 'CRGO': 'ТРАНСПОРТ', 'OPLN': 'ТРАНСПОРТ', 'GSP_T': 'ТРАНСПОРТ',
    'NSTR': 'ТРАНСПОРТ',

    # РИТЕЙЛ (25 тикеров)
    'LENT': 'РИТЕЙЛ', 'MGNT': 'РИТЕЙЛ', 'FIVE': 'РИТЕЙЛ', 'FIXP': 'РИТЕЙЛ', 'MVID': 'РИТЕЙЛ', 
    'OKEY': 'РИТЕЙЛ', 'APTK': 'РИТЕЙЛ', 'LENTP': 'РИТЕЙЛ', 'MGNTP': 'РИТЕЙЛ', 'FIVEP': 'РИТЕЙЛ',
    'VART': 'РИТЕЙЛ', 'SMRK': 'РИТЕЙЛ', 'DISC': 'РИТЕЙЛ', 'ECOM': 'РИТЕЙЛ', 'ORAL': 'РИТЕЙЛ',
    'SPO_R': 'РИТЕЙЛ', 'FSHN': 'РИТЕЙЛ', 'CLTH': 'РИТЕЙЛ', 'GROC': 'РИТЕЙЛ', 'HMRG': 'РИТЕЙЛ',
    'ALLR': 'РИТЕЙЛ', 'MELN': 'РИТЕЙЛ', 'KIDZ': 'РИТЕЙЛ', 'DSKY': 'РИТЕЙЛ', 'OZON_R': 'РИТЕЙЛ',

    # ХИМИЯ (20 тикеров)
    'PHOR': 'ХИМИЯ', 'AKRN': 'ХИМИЯ', 'KAZM': 'ХИМИЯ', 'KZOS': 'ХИМИЯ', 'KZOSP': 'ХИМИЯ',
    'NKNC': 'ХИМИЯ', 'NKNCP': 'ХИМИЯ', 'KAZMP': 'ХИМИЯ', 'AKRNP': 'ХИМИЯ', 'PHORP': 'ХИМИЯ',
    'KHIM': 'ХИМИЯ', 'PLST': 'ХИМИЯ', 'FERT': 'ХИМИЯ', 'URKA': 'ХИМИЯ', 'META': 'ХИМИЯ',
    'ORCH': 'ХИМИЯ', 'BIOH': 'ХИМИЯ', 'AZOT': 'ХИМИЯ', 'KOKS': 'ХИМИЯ', 'SODA': 'ХИМИЯ',

    # СТРОИТЕЛЬСТВО (15 тикеров)
    'PIKK': 'СТРОИТЕЛЬСТВО', 'LSRG': 'СТРОИТЕЛЬСТВО', 'SMLT': 'СТРОИТЕЛЬСТВО', 'PIKKP': 'СТРОИТЕЛЬСТВО',
    'SMLTP': 'СТРОИТЕЛЬСТВО', 'LSRGP': 'СТРОИТЕЛЬСТВО', 'INTR': 'СТРОИТЕЛЬСТВО', 'ETLN': 'СТРОИТЕЛЬСТВО',
    'RENO': 'СТРОИТЕЛЬСТВО', 'DSCT': 'СТРОИТЕЛЬСТВО', 'MSTR': 'СТРОИТЕЛЬСТВО', 'URBL': 'СТРОИТЕЛЬСТВО',
    'BLDG': 'СТРОИТЕЛЬСТВО', 'STYL': 'СТРОИТЕЛЬСТВО', 'SKAR': 'СТРОИТЕЛЬСТВО',

    # ТЕЛЕКОМ (15 тикеров)
    'MTSS': 'ТЕЛЕКОМ', 'RTKM': 'ТЕЛЕКОМ', 'RTKMP': 'ТЕЛЕКОМ', 'CNTL': 'ТЕЛЕКОМ', 'CNTLP': 'ТЕЛЕКОМ',
    'MTSSP': 'ТЕЛЕКОМ', 'MEGA': 'ТЕЛЕКОМ', 'VMPC': 'ТЕЛЕКОМ', 'TTLK': 'ТЕЛЕКОМ', 'SVTG': 'ТЕЛЕКОМ',
    'TELC': 'ТЕЛЕКОМ', 'NETW': 'ТЕЛЕКОМ', 'FBRS': 'ТЕЛЕКОМ', 'SATL': 'ТЕЛЕКОМ', 'WRLS': 'ТЕЛЕКОМ',

    # МЕДИЦИНА (13 тикеров)
    'MDMG': 'МЕДИЦИНА', 'GEMC': 'МЕДИЦИНА', 'MDMGP': 'МЕДИЦИНА', 'GEMCP': 'МЕДИЦИНА', 'PHRM': 'МЕДИЦИНА',
    'CLNC': 'МЕДИЦИНА', 'MEDC': 'МЕДИЦИНА', 'LABS': 'МЕДИЦИНА', 'GENO': 'МЕДИЦИНА', 'PHAR': 'МЕДИЦИНА',
    'VITA': 'МЕДИЦИНА', 'IMUN': 'МЕДИЦИНА', 'DENT': 'МЕДИЦИНА',

    # МАЙНИНГ (12 тикеров)
    'RASP': 'МАЙНИНГ', 'MTLR': 'МАЙНИНГ', 'MTLRP': 'МАЙНИНГ', 'RASPP': 'МАЙНИНГ', 'SGRV': 'МАЙНИНГ',
    'SGRVP': 'МАЙНИНГ', 'ALMN': 'МАЙНИНГ', 'COAL': 'МАЙНИНГ', 'GOLD': 'МАЙНИНГ', 'URMN': 'МАЙНИНГ',
    'SBMN': 'МАЙНИНГ', 'ZAMN': 'МАЙНИНГ',

    # ИНВЕСТИЦИИ (10 тикеров)
    'AFKS': 'ИНВЕСТИЦИИ', 'SFIN': 'ИНВЕСТИЦИИ', 'AFKSP': 'ИНВЕСТИЦИИ', 'SFINP': 'ИНВЕСТИЦИИ',
    'INVC': 'ИНВЕСТИЦИИ', 'VCFT': 'ИНВЕСТИЦИИ', 'ASST': 'ИНВЕСТИЦИИ', 'PRTF': 'ИНВЕСТИЦИИ',
    'EQTY': 'ИНВЕСТИЦИИ', 'SEED': 'ИНВЕСТИЦИИ',

    # ДРУГОЕ (10 тикеров)
    'ALRS': 'ДРУГОЕ', 'SVH': 'ДРУГОЕ', 'ALRSP': 'ДРУГОЕ', 'MISC': 'ДРУГОЕ', 'ABRD': 'ДРУГОЕ',
    'NKSH': 'ДРУГОЕ', 'KROT_D': 'ДРУГОЕ', 'UNKN': 'ДРУГОЕ', 'MXBD': 'ДРУГОЕ', 'HOLD': 'ДРУГОЕ'
}

# Обновление списка всех тикеров на основе ключей словаря
all_tickers = list(ticker_to_sector.keys())


unique_sectors = sorted(list(set(ticker_to_sector.values())))
domains_records = []

sector_to_pid = {}

for idx, sector in enumerate(unique_sectors):
    pid = 100 + idx * 10 
    sector_to_pid[sector] = pid
    
    if sector in ['ФИНАНСЫ', 'НЕФТЬ И ГАЗ', 'ЭНЕРГЕТИКА']:
        criticality = 'High'
    elif sector in ['МЕТАЛЛУРГИЯ', 'ИТ', 'ТРАНСПОРТ']:
        criticality = 'Medium'
    else:
        criticality = 'Low'
        
    domains_records.append({
        'PROJECT_ID': pid,
        'DATA_DOMAIN': sector,
        'CRITICALITY': criticality
    })

df_domains = pd.DataFrame(domains_records)

# ======================================================
# 2. РАСШИРЕННАЯ ИМИТАЦИЯ СИСТЕМНЫХ ЛОГОВ И СБОЕВ 
# ======================================================

start_date = datetime(2026, 5, 10)

loaders_records = []
bugs_records = []
dependencies_records = []

req_id_counter = 50000
bug_id_counter = 1
loader_id_counter = 1

error_templates = [
    # 1. Производительность БД
    "Deadlock в СУБД. Конфликт параллельной блокировки целевой таблицы при записи асинхронного потока.",
    "Timeout промежуточного слоя STAGE. Превышено время ожидания ответа от сервера базы данных.",
    "Индекс базы данных неактивен или поврежден. Время выполнения тяжелого аналитического запроса превысило лимит.",
    "Пул соединений СУБД полностью исчерпан (Connection pool overflow). Запросы на запись встали в очередь.",
    "Критическое падение IOPS на дисковом массиве базы данных. СУБД не успевает сбрасывать логи транзакций на диск.",
    
    # 2. Ошибки в данных/источнике
    "Ошибка валидации DWH. Обнаружено некорректное (отрицательное) значение в поле FACEVALUE пакета.",
    "Неконсистентная структура JSON источника. В ответе API отсутствует обязательный массив 'history.columns'.",
    "Сбой парсинга XML-схемы метаданных. Невалидная кодировка или недопустимый токен в строке конфигурации.",
    "Поставщик прислал пустой файл (размер 0 Кб). Инкрементальная выгрузка за текущий операционный день пуста.",
    "Дублирование первичных ключей (Primary Key Violation) в сыром пакете данных от внешнего шлюза.",
    
    # 3. Проблемы с зависимостями
    "Data Lineage Error. Нарушена ссылочная целостность: EMITTER_ID отсутствует в системном справочнике.",
    "Сбой консистентности. Предыдущий инкремент завершен успешно, но целевой объект пуст. Данные не доехали.",
    "Каскадная блокировка пайплайна: родительская задача в Airflow упала, расчет дочерней витрины отменен.",
    "Сбой схемы данных (Schema Drift). Дочерний процесс ожидает структуру таблицы, которая была изменена в апстриме.",
    "Внешний мастер-справочник заблокирован другим процессом сборки. Доступ к родительским метаданным ограничен.",
    
    # 4. Ошибка в коде/логике
    "DataOverflowError. Значение объема выпуска (ISSUESIZE) превышает лимит типа данных INT в целевой таблице.",
    "ZeroDivisionError в модуле расчета средневзвешенной цены. Деление на ноль при отсутствии сделок в тикерном пакете.",
    "Ошибка трансформации данных: некорректная регулярка в маппинге текстового поля привела к падению парсера.",
    "Исключение ValueError при попытке привести строковое значение даты к формату TIMESTAMP в теле функции.",
    "Логическая ошибка инкремента: фильтр по дате (WHERE) отсек весь массив закрытия сессии из-за сдвига часовых поясов.",
    
    # 5. Инфраструктурные сбои
    "API MOEX ISS недоступен. Ошибка 504 Gateway Timeout. Возможные работы на стороне сетевого контура.",
    "HTTP 429 Too Many Requests. Превышен лимит параллельных запросов к шлюзу Мосбиржи под текущим токеном.",
    "ConnectionResetError. Сессия принудительно разорвана удаленным хостом API. Требуется рестарт таска.",
    "SSL: CERTIFICATE_VERIFY_FAILED. Ошибка проверки сертификата безопасности магистрального шлюза.",
    "Disk Full Error. Недостаточно места на узле хранения для фиксации транзакции. Лог забил системный диск.",
    "Процесс завис на этапе выгрузки. Таймаут Airflow превышен. Сессия завершена принудительно (kill -9).",
    
    # 6. Человеческий фактор
    "Конфликт версий конфигурации. Обнаружен некорректный ручной запуск закрытой сессии поверх автоматического пайплайна.",
    "Сбой аутентификации. Дежурным инженером указан устаревший сервисный токен в параметрах ручного перезапуска.",
    "Пайплайн заблокирован: администратор вручную выставил статус PAUSED для корневого дага в Airflow.",
    "Параметры выгрузки переписаны вручную из консоли с неверным указанием временного диапазона (Date Range mismatch).",
    "Тестовый конфиг ошибочно залит в прод-репозиторий при ручном комите. Пути к сырым данным ведут на staging-сервер."
]
ticker_to_oid = {ticker: 4000 + idx for idx, ticker in enumerate(all_tickers)}
for day in range(20):
    current_day = start_date + timedelta(days=day)
    sla_time = current_day.replace(hour=19, minute=15, second=0, microsecond=0)
    day_req_ids = {}
    
    is_heavy_load_day = (day % 5 == 0)
    
    for ticker in all_tickers:
        req_id_counter += 1
        day_req_ids[ticker] = req_id_counter
        
        rand = random.random()

        if is_heavy_load_day:
            success_threshold = 0.70  # Было 0.82
            failed_threshold = 0.85   # Было 0.93 
        else:
            success_threshold = 0.90  # Было 0.98
            failed_threshold = 0.95 
        # Текущая дата в виде строки для точечных проверок
        current_date_str = current_day.strftime('%Y-%m-%d')
            
        # АНОМАЛИЯ 1: Сбер лагает строго на Неделе 2 (с 11 по 17 мая)
        if '2026-05-11' <= current_date_str <= '2026-05-17' and ticker in ['SBER', 'SBERP', 'SBER_E']:
            success_threshold = 0.30  # Сбер падает/зависает в 70% случаев
            failed_threshold = 0.80
            
        # АНОМАЛИЯ 2: ИТ-сектор штормит строго на Неделе 2 (с 11 по 17 мая)
        elif '2026-05-11' <= current_date_str <= '2026-05-17' and ticker_to_sector[ticker] == 'ИТ':
            success_threshold = 0.65  # ИТ-сектор падает в 50% случаев
            failed_threshold = 0.85
        # Базовое время старта (по умолчанию для SUCCESS процессов)
        begin_time = current_day.replace(
            hour=19, 
            minute=0, 
            second=random.randint(0, 59), 
            microsecond=random.randint(0, 999999)
        )
        
        if rand < success_threshold:
            status = 'SUCCESS'
            max_minute = 12 if is_heavy_load_day else 10
            end_time = current_day.replace(hour=19, minute=random.randint(2, max_minute), second=random.randint(0, 59))
            
            if day == 12:
                # Допустим, диск переполнился ровно в 19:06:00
                disk_full_time = current_day.replace(hour=19, minute=6, second=0)
                
                if end_time >= disk_full_time:
                    # Процесс не успел записаться до переполнения диска!
                    status = 'FAILED'
                    # Время фиксации ошибки системой — чуть позже времени аварии
                    end_time = disk_full_time + timedelta(seconds=random.randint(5, 30))
        
        elif rand < failed_threshold:
            status = 'FAILED'
            fail_hour = random.choice([11, 13, 14, 15, 16])
            begin_time = current_day.replace(hour=fail_hour, minute=random.randint(10, 50), second=random.randint(0, 59))
            end_time = current_day.replace(hour=19, minute=random.randint(1, 10), second=random.randint(0, 59))
        
        else:
            status = 'RUNNING'
            fail_hour = random.choice([12, 14, 15, 17])
            begin_time = current_day.replace(hour=fail_hour, minute=random.randint(10, 50), second=random.randint(0, 59))
            end_time = np.nan
            
        current_now = datetime.now() + timedelta(microseconds=random.randint(0, 1000))
            
        loaders_records.append({
            'ID': loader_id_counter,
            'REQ_ID': req_id_counter,
            'REQ_TYPE': 'Daily',
            'REQ_STATUS': status,
            'REQ_BEGIN_DATE': begin_time,
            'REQ_END_DATE': end_time,
            'UPDATEDT': current_now
        })
        
       # Если статус сбойный - создаем детальный инцидент
        if status in ['FAILED', 'RUNNING']:
            sector = ticker_to_sector[ticker]
            pid = sector_to_pid[sector]
            object_name = f"STAGE_MOEX_{ticker}_RAW"
            
            # 1. Базовый выбор ошибки по умолчанию (для обычных дней)
            base_err = random.choice(error_templates)
            
            # =====================================================================
            # ШАГ А: АНОМАЛИИ СЕКТОРОВ И ТИКЕРОВ (ДЛЯ ОБЫЧНЫХ ДНЕЙ)
            # =====================================================================

            # 1. СИСТЕМНАЯ ПРОБЛЕМА СБЕРА (Лагает стабильно всю неделю, например, с 11 по 17 мая)
            if '2026-05-11' <= current_date_str <= '2026-05-17' and ticker in ['SBER', 'SBERP', 'SBER_E']:
                # Вместо редких падений, заставляем СУБД ловить Deadlock с вероятностью 45% каждый день этой недели
                if random.random() < 0.95:
                    base_err = "Deadlock в СУБД. Конфликт параллельной блокировки целевой таблицы при записи асинхронного потока."
                    # Чтобы это влияло на SLA, можно принудительно выставить флаг задержки в логике ниже, если требуется

            # 2. СИСТЕМНЫЙ СБОЙ ИТ-СЕКТОРА (Полмесяца ловит ошибки чаще, например, с 10 по 25 мая)
            elif '2026-05-11' <= current_date_str <= '2026-05-17' and sector == 'ИТ':
                # Повышаем базовую частоту ошибок для ИТ-сектора. 
                # В обычные дни вероятность сбоя условные 5%, а в эти полмесяца — стабильные 35%
                if random.random() < 0.7:
                    base_err = random.choice([
                        "API MOEX ISS недоступен. Ошибка 504 Gateway Timeout. Возможные работы на стороне сетевого контура.",
                        "HTTP 429 Too Many Requests. Превышен лимит параллельных запросов к шлюзу Мосбиржи под текущим токеном."
            ])

            # =====================================================================
            # ШАГ Б: РАСЧЕТ SLA, FIX_DATE И ТЕКСТОВ С УЧЕТОМ ДНЯ ИКС
            # =====================================================================
            
            # --- ОСОБАЯ ЛОГИКА ДЛЯ ДНЯ ИКС (ДЕНЬ 12: ПЕРЕПОЛНЕНИЕ ДИСКА) ---
            if day == 12:
                base_err = "Disk Full Error. Недостаточно места на узле хранения для фиксации транзакции. Лог забил системный диск."
                
                # Разделяем на вечерний шторм (после 19:06) и случайные дневные падения
                is_evening_storm = (begin_time.hour == 19 and begin_time.minute == 0)
                
                if is_evening_storm:
                    # Вечерние процессы, которые упали строго в момент аварии (19:06)
                    is_sla_fall = 1 if random.random() < 0.50 else 0  # 50% пробивают регламент
                    
                    if is_sla_fall == 1:
                        # Не успели починить до 19:15
                        fix_date = end_time + timedelta(minutes=random.randint(25, 55)) # починка в районе 19:31 - 20:01
                        bug_details = f"{base_err} Target Object: [{object_name}]. Критическая авария в 19:06:00. Нарушен регламент поставки (SLA)."
                        fix_desc = "Автоматический рестарт заблокирован ОС. Выделен дополнительный дисковый массив. Проведена очистка логов WAL. Пайплайн переведен на резервный узел."
                    else:
                        # Успели оперативно «пнуть» и очистить место до 19:15!
                        fix_date = end_time + timedelta(minutes=random.randint(4, 8)) # починка строго до 19:14
                        bug_details = f"{base_err} Target Object: [{object_name}]. Локальный сбой записи инкремента."
                        fix_desc = "Дежурный инженер оперативно очистил кэш системы (/tmp space). Место освобождено, таск успешно перезапущен до наступления SLA."
                
                else:
                    # Дневные случайные падения, которые произошли ДО аварии диска (например, в 11:00)
                    # Но так как диск переполнился в 19:06, их исправление (fix_date) застряло и улетело на вечер!
                    is_sla_fall = 1 
                    fix_date = current_day.replace(hour=19, minute=random.randint(30, 50)) # Починят только вечером вместе со всеми
                    bug_details = f"{base_err} Target Object: [{object_name}]. Дневной таск заблокирован каскадным падением дисковой подсистемы в 19:06."
                    fix_desc = "Восстановление отложено из-за инфраструктурного шторма. Данные перезалиты вручную после расширения дискового пула."

            # --- СТАНДАРТНАЯ ЛОГИКА ДЛЯ ВСЕХ ОСТАЛЬНЫХ ДНЕЙ ---
            else:
                # СЦЕНАРИЙ 1: ПРОЦЕСС ЗАВИС (RUNNING)
                if pd.isna(end_time):  
                    is_sla_fall = 1 if random.random() < 0.25 else 0 
                    if ticker in ['SBER', 'SBERP']:
                        is_sla_fall = 1

                    if is_sla_fall == 1:
                        fix_date = current_day.replace(hour=19, minute=random.randint(35, 55))
                        bug_details = f"Процесс загрузки {ticker} критически завис. Первое аномальное поведение зафиксировано в {begin_time.strftime('%H:%M')}. Превышен таймаут сессии Airflow."
                        fix_desc = f"Принудительное завершение процесса администратором (kill -9) в {fix_date.strftime('%H:%M')} и ручной перезапуск пайплайна."
                    else:
                        fix_date = begin_time + timedelta(minutes=random.randint(60, 120))
                        bug_details = f"Процесс загрузки {ticker} временно завис в дневную смену. Зафиксирована потеря пакетов промежуточного слоя."
                        fix_desc = "Зависший процесс обнаружен и принудительно перезапущен дежурным администратором днем."

                # СЦЕНАРИЙ 2: ПРОЦЕСС УПАЛ (FAILED)
                else:  
                    is_sla_fall = 1 if random.random() < 0.25 else 0 
                    if ticker in ['SBER', 'SBERP']:
                        is_sla_fall = 1

                    if is_sla_fall == 1:
                        end_time = current_day.replace(hour=19, minute=random.randint(16, 25))
                        loaders_records[-1]['REQ_END_DATE'] = end_time  
                        fix_date = end_time + timedelta(minutes=random.randint(20, 45))
                        bug_details = f"{base_err} Target Object: [{object_name}]. Нарушен регламент поставки (SLA)."
                        fix_desc = "Автоматический перезапуск заблокирован. Потребовался ручной откат транзакций в СУБД."
                    else:
                        fix_date = end_time + timedelta(minutes=random.randint(1, 5))
                        # Страховка: если дневной фикс случайно вылез за 19:15, принудительно двигаем его обратно, чтобы не ломать логику SLA=0
                        if fix_date > sla_time:
                            fix_date = sla_time - timedelta(minutes=random.randint(5, 12))
                        bug_details = f"{base_err} Target Object: [{object_name}]. Сбой локализован в дневную смену."
                        fix_desc = "Сработал автоматический Retry в Airflow. Данные успешно догружены, пайплайн стабилизирован."

            # =====================================================================
            # ШАГ В: ВЫЧИСЛЕНИЕ СИСТЕМНЫХ МЕТАДАННЫХ (СЛУЖЕБНЫЕ ДАТЫ, ПРИОРИТЕТЫ)
            # =====================================================================
            
            # Настройка дат просрочек для SLA и прогнозов починки (ETA)
            if is_sla_fall == 1:
                sla_fall_date = sla_time
                eta_fall_date = sla_time + timedelta(minutes=random.randint(15, 30))
            else:
                sla_fall_date = pd.NaT
                eta_fall_date = pd.NaT

            # Настройка приоритетов на основе критичности домена данных
            if sector in ['ФИНАНСЫ', 'НЕФТЬ И ГАЗ', 'ЭНЕРГЕТИКА']:
                priority = 'High' if is_sla_fall == 1 else 'Medium'
            elif sector in ['МЕТАЛЛУРГИЯ', 'ИТ', 'ТРАНСПОРТ']:
                priority = 'Medium' if is_sla_fall == 1 else 'Low'
            else:
                priority = 'Low'

            # Запись инцидента в массив
            bugs_records.append({
                'ID': bug_id_counter,
                'PID': pid,
                'REQ_ID': req_id_counter,
                'PROJECT_NAME': ticker,
                'OID': ticker_to_oid[ticker],
                'OBJECT_NAME': object_name,
                'SLA_FALL': is_sla_fall,
                'SLA_FALL_DATE': sla_fall_date,  
                'ETA_FALL_DATE': eta_fall_date,   
                'PRIORITY': priority,
                'CREATE_DATE': begin_time,   
                'FIX_DATE': fix_date,         
                'BUG_DETAILS': bug_details,
                'FIX_DESCRIPTION': fix_desc,
                'RESPONSIBLE': random.choice(['Терпунов А.А.','Гайсин К.Д.', 'Куликов Н.С.']),
                'UPDATEDT': current_now
            })
            bug_id_counter += 1
            
        loader_id_counter += 1
        
    # КОРНЕВЫЕ ЗАВИСИМОСТИ (Генерируются каждый день для создания каскадных цепочек сбоев)
    dependencies_records.append({
        'I_REQ_ID': day_req_ids['LKOH'],
        'UP_REQ_ID': day_req_ids['GAZP'],
        'I_SCHEMA_NAME': 'STAGE_MOEX',
        'COMMENTS': f"Каскадная зависимость: Расчет витрин Лукойла требует предварительной фиксации сырых данных Газпрома от {current_day.strftime('%Y-%m-%d')}."
    })
    #В финансовом секторе VTBR зависит от SBER
    dependencies_records.append({
        'I_REQ_ID': day_req_ids['VTBR'],
        'UP_REQ_ID': day_req_ids['SBER'],
        'I_SCHEMA_NAME': 'STAGE_MOEX',
        'COMMENTS': f"Секторальная зависимость: Межбанковская аналитика ВТБ ожидает выгрузку реестра Сбербанка."
    })

df_loaders = pd.DataFrame(loaders_records)
df_bugs = pd.DataFrame(bugs_records)
df_dependencies = pd.DataFrame(dependencies_records)

# =====================================================================
# 3. КОНТРОЛЬНЫЙ ВЫВОД РЕЗУЛЬТАТОВ И СОХРАНЕНИЕ В CSV
# =====================================================================

print("="*60)
print(" СТАТИСТИКА СГЕНЕРИРОВАННОГО ДАТАСЕТА ")
print("="*60)
print(f"1. Строк в Таблице Доменов      (df_domains):      {len(df_domains)}")
print(f"2. Строк в Таблице Загрузчиков  (df_loaders):      {len(df_loaders)}")
print(f"3. Строк в Таблице Инцидентов  (df_bugs):         {len(df_bugs)}")
print(f"4. Строк в Таблице Зависимостей (df_dependencies): {len(df_dependencies)}")
print("="*60)


# РАСКОММЕНТИРУЙ СТРОКИ НИЖЕ, ЧТОБЫ СОХРАНИТЬ ДАННЫЕ В ФАЙЛЫ
df_domains.to_csv('table_domains.csv', index=False)
df_loaders.to_csv('table_loaders.csv', index=False)
df_bugs.to_csv('table_bugs.csv', index=False)
df_dependencies.to_csv('table_dependencies.csv', index=False)
print("\n[INFO] Файлы успешно сохранены в CSV для дальнейшей обработки!")

 СТАТИСТИКА СГЕНЕРИРОВАННОГО ДАТАСЕТА 
1. Строк в Таблице Доменов      (df_domains):      14
2. Строк в Таблице Загрузчиков  (df_loaders):      6300
3. Строк в Таблице Инцидентов  (df_bugs):         1112
4. Строк в Таблице Зависимостей (df_dependencies): 40

[INFO] Файлы успешно сохранены в CSV для дальнейшей обработки!


In [17]:
display(df_domains)

,PROJECT_ID,DATA_DOMAIN,CRITICALITY
0,100,ДРУГОЕ,Low
1,110,ИНВЕСТИЦИИ,Low
2,120,ИТ,Medium
3,130,МАЙНИНГ,Low
4,140,МЕДИЦИНА,Low
5,150,МЕТАЛЛУРГИЯ,Medium
6,160,НЕФТЬ И ГАЗ,High
7,170,РИТЕЙЛ,Low
8,180,СТРОИТЕЛЬСТВО,Low
9,190,ТЕЛЕКОМ,Low


In [18]:
display(df_loaders)

,ID,REQ_ID,REQ_TYPE,REQ_STATUS,REQ_BEGIN_DATE,REQ_END_DATE,UPDATEDT
0,1,50001,Daily,SUCCESS,2026-05-10 19:00:01.777572,2026-05-10 19:06:15,2026-06-08 07:04:19.877435
1,2,50002,Daily,SUCCESS,2026-05-10 19:00:06.709570,2026-05-10 19:10:05,2026-06-08 07:04:19.877811
2,3,50003,Daily,SUCCESS,2026-05-10 19:00:01.098246,2026-05-10 19:05:14,2026-06-08 07:04:19.877724
3,4,50004,Daily,SUCCESS,2026-05-10 19:00:35.208496,2026-05-10 19:12:44,2026-06-08 07:04:19.877765
4,5,50005,Daily,SUCCESS,2026-05-10 19:00:28.617889,2026-05-10 19:06:51,2026-06-08 07:04:19.878097
...,...,...,...,...,...,...,...
6295,6296,56296,Daily,SUCCESS,2026-05-29 19:00:46.556461,2026-05-29 19:10:32,2026-06-08 07:04:19.952345
6296,6297,56297,Daily,SUCCESS,2026-05-29 19:00:30.536593,2026-05-29 19:08:03,2026-06-08 07:04:19.953221
6297,6298,56298,Daily,SUCCESS,2026-05-29 19:00:59.848295,2026-05-29 19:04:07,2026-06-08 07:04:19.952626
6298,6299,56299,Daily,SUCCESS,2026-05-29 19:00:09.523387,2026-05-29 19:06:13,2026-06-08 07:04:19.952718


In [19]:
display(df_bugs)

,ID,PID,REQ_ID,PROJECT_NAME,OID,OBJECT_NAME,SLA_FALL,SLA_FALL_DATE,ETA_FALL_DATE,PRIORITY,CREATE_DATE,FIX_DATE,BUG_DETAILS,FIX_DESCRIPTION,RESPONSIBLE,UPDATEDT
0,1,160,50011,RNFT,4010,STAGE_MOEX_RNFT_RAW,0,NaT,NaT,Medium,2026-05-10 12:12:42,2026-05-10 13:17:42,Процесс загрузки RNFT временно завис в дневную...,Зависший процесс обнаружен и принудительно пер...,Терпунов А.А.,2026-06-08 07:04:19.877440
1,2,160,50012,GAZPP,4011,STAGE_MOEX_GAZPP_RAW,0,NaT,NaT,Medium,2026-05-10 17:50:53,2026-05-10 19:03:53,Процесс загрузки GAZPP временно завис в дневну...,Зависший процесс обнаружен и принудительно пер...,Куликов Н.С.,2026-06-08 07:04:19.877580
2,3,160,50018,ZNTO,4017,STAGE_MOEX_ZNTO_RAW,0,NaT,NaT,Medium,2026-05-10 15:23:41,2026-05-10 17:04:41,Процесс загрузки ZNTO временно завис в дневную...,Зависший процесс обнаружен и принудительно пер...,Гайсин К.Д.,2026-06-08 07:04:19.879230
3,4,160,50020,REFT,4019,STAGE_MOEX_REFT_RAW,1,2026-05-10 19:15:00,2026-05-10 19:34:00,High,2026-05-10 16:35:23,2026-05-10 19:39:00,DataOverflowError. Значение объема выпуска (IS...,Автоматический перезапуск заблокирован. Потреб...,Куликов Н.С.,2026-06-08 07:04:19.879240
4,5,160,50023,ASTN,4022,STAGE_MOEX_ASTN_RAW,0,NaT,NaT,Medium,2026-05-10 12:44:48,2026-05-10 13:51:48,Процесс загрузки ASTN временно завис в дневную...,Зависший процесс обнаружен и принудительно пер...,Гайсин К.Д.,2026-06-08 07:04:19.878992
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1107,1108,220,56210,KZOSP,4224,STAGE_MOEX_KZOSP_RAW,0,NaT,NaT,Low,2026-05-29 15:14:27,2026-05-29 16:24:27,Процесс загрузки KZOSP временно завис в дневну...,Зависший процесс обнаружен и принудительно пер...,Терпунов А.А.,2026-06-08 07:04:19.951575
1108,1109,220,56218,FERT,4232,STAGE_MOEX_FERT_RAW,1,2026-05-29 19:15:00,2026-05-29 19:31:00,Low,2026-05-29 12:47:45,2026-05-29 19:47:00,Процесс загрузки FERT критически завис. Первое...,Принудительное завершение процесса администрат...,Гайсин К.Д.,2026-06-08 07:04:19.951646
1109,1110,180,56228,SMLT,4242,STAGE_MOEX_SMLT_RAW,0,NaT,NaT,Low,2026-05-29 13:39:16,2026-05-29 19:11:25,Логическая ошибка инкремента: фильтр по дате (...,Сработал автоматический Retry в Airflow. Данны...,Куликов Н.С.,2026-06-08 07:04:19.951692
1110,1111,190,56248,VMPC,4262,STAGE_MOEX_VMPC_RAW,0,NaT,NaT,Low,2026-05-29 15:34:53,2026-05-29 17:21:53,Процесс загрузки VMPC временно завис в дневную...,Зависший процесс обнаружен и принудительно пер...,Куликов Н.С.,2026-06-08 07:04:19.951284


In [20]:
display(df_dependencies)

,I_REQ_ID,UP_REQ_ID,I_SCHEMA_NAME,COMMENTS
0,50002,50001,STAGE_MOEX,Каскадная зависимость: Расчет витрин Лукойла т...
1,50041,50040,STAGE_MOEX,Секторальная зависимость: Межбанковская аналит...
2,50317,50316,STAGE_MOEX,Каскадная зависимость: Расчет витрин Лукойла т...
3,50356,50355,STAGE_MOEX,Секторальная зависимость: Межбанковская аналит...
4,50632,50631,STAGE_MOEX,Каскадная зависимость: Расчет витрин Лукойла т...
5,50671,50670,STAGE_MOEX,Секторальная зависимость: Межбанковская аналит...
6,50947,50946,STAGE_MOEX,Каскадная зависимость: Расчет витрин Лукойла т...
7,50986,50985,STAGE_MOEX,Секторальная зависимость: Межбанковская аналит...
8,51262,51261,STAGE_MOEX,Каскадная зависимость: Расчет витрин Лукойла т...
9,51301,51300,STAGE_MOEX,Секторальная зависимость: Межбанковская аналит...


In [21]:
import pandas as pd

def process_sla_falls(df_loaders: pd.DataFrame, df_bugs: pd.DataFrame, df_domains: pd.DataFrame) -> pd.DataFrame:
    # 1. За основу берем ВСЕ загрузчики
    df_res = df_loaders.copy()
    
    df_res['REQ_BEGIN_DATE'] = pd.to_datetime(df_res['REQ_BEGIN_DATE'])
    df_res['REQ_END_DATE'] = pd.to_datetime(df_res['REQ_END_DATE'])
    
    # 2. Подготавливаем данные по багам
    df_bugs_prep = df_bugs.copy()
    df_bugs_prep['SLA_FALL_DATE'] = pd.to_datetime(df_bugs_prep['SLA_FALL_DATE'])
    df_bugs_prep['CREATE_DATE'] = pd.to_datetime(df_bugs_prep['CREATE_DATE'])
    df_bugs_prep['FIX_DATE'] = pd.to_datetime(df_bugs_prep['FIX_DATE'])
    
    # Расчет времени фикса багов
    current_time = pd.Timestamp.now()
    end_time = df_bugs_prep['FIX_DATE'].fillna(current_time)
    df_bugs_prep['FIX_TIME_HOURS'] = ((end_time - df_bugs_prep['CREATE_DATE']).dt.total_seconds() / 3600.0).round(2)
    
    # Очищаем баги от дубликатов по REQ_ID перед объединением
    df_bugs_clean = df_bugs_prep.drop_duplicates(subset=['REQ_ID'], keep='last')

    # 3. LEFT JOIN: Соединяем лоадеры с багами
    df_res = df_res.merge(
        df_bugs_clean[[
            'REQ_ID', 'ID', 'PROJECT_NAME', 'OBJECT_NAME', 'SLA_FALL', 'SLA_FALL_DATE', 
            'PRIORITY', 'BUG_DETAILS', 'FIX_DESCRIPTION', 'RESPONSIBLE', 'CREATE_DATE', 
            'FIX_DATE', 'FIX_TIME_HOURS'
        ]], 
        on='REQ_ID', 
        how='left',
        suffixes=('_LOADER', '_BUG')
    )
    
    df_res = df_res.rename(columns={'ID_BUG': 'BUG_ID', 'ID_LOADER': 'ID'})

    # =====================================================================
    # ШАГ 4: СВЯЗЫВАНИЕ ДОМЕНОВ ДЛЯ ВСЕХ СТРОК (ВКЛЮЧАЯ SUCCESS)
    # =====================================================================
    
    # Из генератора данных у нас есть глобальный словарь ticker_to_sector.
    # Сделаем из него DataFrame, чтобы применить merge.
    df_ticker_sector = pd.DataFrame(list(ticker_to_sector.items()), columns=['TICKER', 'SECTOR'])
    
    # Объединяем тикеры со справочником доменов, чтобы получить их CRITICALITY
    df_full_domain_map = df_ticker_sector.merge(
        df_domains[['DATA_DOMAIN', 'CRITICALITY']], 
        left_on='SECTOR', 
        right_on='DATA_DOMAIN', 
        how='left'
    )
    # Теперь у нас есть четкая карта: TICKER -> DATA_DOMAIN, CRITICALITY
    
    # Но в df_loaders изначально нет тикера! Зато у нас есть PROJECT_NAME из df_bugs для ошибок.
    # А для SUCCESS-строк мы можем вытащить тикер (PROJECT_NAME), зная, что в твоем датасете 
    # REQ_ID жестко привязан к тикерам циклически. 
    # Самый надежный способ восстановить PROJECT_NAME для SUCCESS — взять маппинг REQ_ID -> PROJECT_NAME из всей таблицы багов.
    # Если же сессия успешная и в багах ее нет, то зная, что на каждый день генерируется ровно len(all_tickers) записей,
    # мы можем определить тикер по остатку от деления:
    
    # Создаем маппинг REQ_ID -> Тикер на основе известных багов
    known_bugs_map = df_bugs_clean.set_index('REQ_ID')['PROJECT_NAME'].to_dict()
    
    # Функция для восстановления тикера (PROJECT_NAME) для абсолютно любой строки
    def recover_project_name(row):
        if pd.notna(row['PROJECT_NAME']):
            return row['PROJECT_NAME']
        # Если в багах нет, восстанавливаем по REQ_ID из логики генератора
        # Смещение: req_id стартовал с 50000. В каждый день day шло по очереди len(all_tickers) штук
        req_id = row['REQ_ID']
        if req_id in known_bugs_map:
            return known_bugs_map[req_id]
        else:
            idx = (req_id - 50001) % len(all_tickers)
            return all_tickers[idx]

    # Прописываем PROJECT_NAME (тикер) для ВСЕХ строк
    df_res['PROJECT_NAME'] = df_res.apply(recover_project_name, axis=1)
    
    # И вот теперь спокойно джоиним домены ко ВСЕМ строкам по восстановленному PROJECT_NAME
    df_res = df_res.merge(
        df_full_domain_map[['TICKER', 'DATA_DOMAIN', 'CRITICALITY']],
        left_on='PROJECT_NAME',
        right_on='TICKER',
        how='left'
    )
    
    # Коалесценция для доменов и критичности
    df_res['DATA_DOMAIN'] = df_res['DATA_DOMAIN'].fillna('НЕОПРЕДЕЛЕНО')
    
    # Если это сбой, приоритет берем из багов. Если SUCCESS — дефолтим на CRITICALITY домена
    df_res['DOMAIN_CRITICALITY'] = df_res['CRITICALITY'].fillna(df_res['PRIORITY'])
    # Для SUCCESS строк, у которых нет PRIORITY, пропишем критичность домена напрямую
    df_res['DOMAIN_CRITICALITY'] = df_res['DOMAIN_CRITICALITY'].fillna(df_res['CRITICALITY'])
    
    # Также восстановим OBJECT_NAME для успешных строк, раз у нас теперь есть тикер
    df_res['OBJECT_NAME'] = df_res['OBJECT_NAME'].fillna("STAGE_MOEX_" + df_res['PROJECT_NAME'] + "_RAW")

    # =====================================================================
    
    # 5. Вычисляем WEEK_START и собираем финальный DF
    df_res['WEEK_START'] = df_res['REQ_BEGIN_DATE'].dt.to_period('W').dt.start_time.dt.date
    
    columns_to_select = [
        'ID', 'REQ_STATUS', 'REQ_BEGIN_DATE', 'REQ_END_DATE', 
        'BUG_ID', 'SLA_FALL', 'PROJECT_NAME', 'OBJECT_NAME', 'SLA_FALL_DATE', 'PRIORITY', 
        'BUG_DETAILS', 'FIX_DESCRIPTION', 'RESPONSIBLE', 'CREATE_DATE', 'FIX_DATE', 
        'DATA_DOMAIN', 'FIX_TIME_HOURS', 'WEEK_START', 'DOMAIN_CRITICALITY'
    ]
    
    final_df = df_res[columns_to_select].sort_values(by='REQ_BEGIN_DATE', ascending=False)
    
    return final_df

# ==========================================
# ВЫЗОВ
# ==========================================
df_end = process_sla_falls(df_loaders, df_bugs, df_domains)
df_end.to_csv('table_end_v3.csv', index=False)
display(df_end)

,ID,REQ_STATUS,REQ_BEGIN_DATE,REQ_END_DATE,BUG_ID,SLA_FALL,PROJECT_NAME,OBJECT_NAME,SLA_FALL_DATE,PRIORITY,BUG_DETAILS,FIX_DESCRIPTION,RESPONSIBLE,CREATE_DATE,FIX_DATE,DATA_DOMAIN,FIX_TIME_HOURS,WEEK_START,DOMAIN_CRITICALITY
6031,6032,SUCCESS,2026-05-29 19:00:59.890548,2026-05-29 19:05:31,NaN,NaN,RENI,STAGE_MOEX_RENI_RAW,NaT,NaN,NaN,NaN,NaN,NaT,NaT,ФИНАНСЫ,NaN,2026-05-25,High
6297,6298,SUCCESS,2026-05-29 19:00:59.848295,2026-05-29 19:04:07,NaN,NaN,UNKN,STAGE_MOEX_UNKN_RAW,NaT,NaN,NaN,NaN,NaN,NaT,NaT,ДРУГОЕ,NaN,2026-05-25,Low
6071,6072,SUCCESS,2026-05-29 19:00:59.756582,2026-05-29 19:02:31,NaN,NaN,RUALP,STAGE_MOEX_RUALP_RAW,NaT,NaN,NaN,NaN,NaN,NaT,NaT,МЕТАЛЛУРГИЯ,NaN,2026-05-25,Medium
6204,6205,SUCCESS,2026-05-29 19:00:59.494072,2026-05-29 19:09:23,NaN,NaN,OZON_R,STAGE_MOEX_OZON_R_RAW,NaT,NaN,NaN,NaN,NaN,NaT,NaT,РИТЕЙЛ,NaN,2026-05-25,Low
6262,6263,SUCCESS,2026-05-29 19:00:59.120331,2026-05-29 19:08:59,NaN,NaN,LABS,STAGE_MOEX_LABS_RAW,NaT,NaN,NaN,NaN,NaN,NaT,NaT,МЕДИЦИНА,NaN,2026-05-25,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242,243,FAILED,2026-05-10 11:31:06.000000,2026-05-10 19:09:43,78.0,0.0,SMLT,STAGE_MOEX_SMLT_RAW,NaT,Low,Дублирование первичных ключей (Primary Key Vio...,Сработал автоматический Retry в Airflow. Данны...,Терпунов А.А.,2026-05-10 11:31:06,2026-05-10 19:14:43,СТРОИТЕЛЬСТВО,7.73,2026-05-04,Low
201,202,FAILED,2026-05-10 11:30:38.000000,2026-05-10 19:03:04,64.0,0.0,APTK,STAGE_MOEX_APTK_RAW,NaT,Low,Поставщик прислал пустой файл (размер 0 Кб). И...,Сработал автоматический Retry в Airflow. Данны...,Куликов Н.С.,2026-05-10 11:30:38,2026-05-10 19:08:04,РИТЕЙЛ,7.62,2026-05-04,Low
80,81,FAILED,2026-05-10 11:25:30.000000,2026-05-10 19:10:54,27.0,0.0,KAZT,STAGE_MOEX_KAZT_RAW,NaT,Low,Индекс базы данных неактивен или поврежден. Вр...,Сработал автоматический Retry в Airflow. Данны...,Куликов Н.С.,2026-05-10 11:25:30,2026-05-10 19:07:00,МЕТАЛЛУРГИЯ,7.69,2026-05-04,Medium
265,266,FAILED,2026-05-10 11:13:15.000000,2026-05-10 19:07:40,84.0,0.0,TELC,STAGE_MOEX_TELC_RAW,NaT,Low,Логическая ошибка инкремента: фильтр по дате (...,Сработал автоматический Retry в Airflow. Данны...,Гайсин К.Д.,2026-05-10 11:13:15,2026-05-10 19:08:40,ТЕЛЕКОМ,7.92,2026-05-04,Low


In [23]:
# =====================================================================
# ДОБАВЛЕНИЕ ROOT CAUSE / КАСКАДНЫХ СЦЕНАРИЕВ ДЛЯ ПРОВЕРКИ ДАШБОРДА
# =====================================================================
# Логика:
# I_REQ_ID  = upstream-реквест, который должен выполниться раньше
# UP_REQ_ID = downstream-реквест, который зависит от I_REQ_ID
#
# Если I_REQ_ID падает, то UP_REQ_ID может стать downstream-инцидентом.
# Это нужно, чтобы дашборд мог показать:
# "какой объект потянул за собой другие проблемы".

import pandas as pd
import numpy as np
from datetime import timedelta

df_bugs_root = df_bugs.copy()
df_loaders_root = df_loaders.copy()
df_dependencies_root = df_dependencies.copy()

# ---------------------------------------------------------------------
# 1. Приведение дат
# ---------------------------------------------------------------------

for col in ["CREATE_DATE", "FIX_DATE", "SLA_FALL_DATE", "ETA_FALL_DATE", "UPDATEDT"]:
    if col in df_bugs_root.columns:
        df_bugs_root[col] = pd.to_datetime(df_bugs_root[col], errors="coerce")

for col in ["REQ_BEGIN_DATE", "REQ_END_DATE", "UPDATEDT"]:
    if col in df_loaders_root.columns:
        df_loaders_root[col] = pd.to_datetime(df_loaders_root[col], errors="coerce")

df_bugs_root["BUG_DAY"] = df_bugs_root["CREATE_DATE"].dt.date

# ---------------------------------------------------------------------
# 2. Справочники по тикерам
# ---------------------------------------------------------------------

ticker_pid = (
    df_bugs_root
    .dropna(subset=["PROJECT_NAME"])
    .drop_duplicates("PROJECT_NAME")
    .set_index("PROJECT_NAME")["PID"]
    .to_dict()
)

ticker_oid = (
    df_bugs_root
    .dropna(subset=["PROJECT_NAME"])
    .drop_duplicates("PROJECT_NAME")
    .set_index("PROJECT_NAME")["OID"]
    .to_dict()
)

ticker_object = (
    df_bugs_root
    .dropna(subset=["PROJECT_NAME"])
    .drop_duplicates("PROJECT_NAME")
    .set_index("PROJECT_NAME")["OBJECT_NAME"]
    .to_dict()
)

max_bug_id = int(df_bugs_root["ID"].max()) + 1
max_loader_id = int(df_loaders_root["ID"].max()) + 1
max_req_id = int(df_loaders_root["REQ_ID"].max()) + 1

# ---------------------------------------------------------------------
# 3. Каскадные сценарии
# ---------------------------------------------------------------------

cascade_cases = [
    {
        "day": pd.to_datetime("2026-05-12").date(),
        "root": "SBER",
        "children": ["VTBR", "MOEX", "TCSG", "SBERP"],
        "root_reason": "Root Cause: сбой базовой финансовой загрузки SBER. Нарушена upstream-витрина финансового домена.",
        "child_reason": "Cascade Failure: downstream-реквест не выполнен из-за сбоя upstream SBER.",
        "schema": "STAGE_MOEX",
        "domain_comment": "Финансовая каскадная зависимость"
    },
    {
        "day": pd.to_datetime("2026-05-13").date(),
        "root": "GAZP",
        "children": ["LKOH", "NVTK", "ROSN", "TATN"],
        "root_reason": "Root Cause: инфраструктурный сбой загрузки GAZP. Базовый нефтегазовый upstream-процесс не завершился.",
        "child_reason": "Cascade Failure: downstream-реквест нефтегазового блока ожидает успешную загрузку GAZP.",
        "schema": "STAGE_MOEX",
        "domain_comment": "Нефтегазовая каскадная зависимость"
    },
    {
        "day": pd.to_datetime("2026-05-14").date(),
        "root": "YNDX",
        "children": ["OZON", "VKCO", "POSI"],
        "root_reason": "Root Cause: API-сбой загрузки YNDX. Upstream-данные ИТ-сектора не были доставлены вовремя.",
        "child_reason": "Cascade Failure: downstream-реквест ИТ-сектора ожидает успешную загрузку YNDX.",
        "schema": "STAGE_MOEX",
        "domain_comment": "ИТ-каскадная зависимость"
    },
    {
        "day": pd.to_datetime("2026-05-15").date(),
        "root": "GMKN",
        "children": ["NLMK", "MAGN", "CHMF"],
        "root_reason": "Root Cause: сбой загрузки GMKN. Upstream-процесс металлургического блока не завершился.",
        "child_reason": "Cascade Failure: downstream-реквест металлургического блока ожидает успешную загрузку GMKN.",
        "schema": "STAGE_MOEX",
        "domain_comment": "Металлургическая каскадная зависимость"
    },
]

new_bugs = []
new_loaders = []
new_dependencies = []


def get_or_create_incident(ticker: str, bug_day, reason: str, role: str) -> int:
    """
    Возвращает REQ_ID инцидента.
    Если инцидент по тикеру в этот день уже есть — используем его.
    Если нет — создаём новый тестовый инцидент.
    """

    global max_bug_id, max_loader_id, max_req_id

    existing = df_bugs_root[
        (df_bugs_root["PROJECT_NAME"] == ticker)
        & (df_bugs_root["BUG_DAY"] == bug_day)
    ]

    if not existing.empty:
        return int(existing.iloc[0]["REQ_ID"])

    create_date = pd.Timestamp(bug_day).replace(hour=18, minute=45)
    sla_time = pd.Timestamp(bug_day).replace(hour=19, minute=15)

    if role == "root":
        req_end_date = pd.Timestamp(bug_day).replace(hour=19, minute=25)
        fix_date = pd.Timestamp(bug_day).replace(hour=20, minute=10)
        priority = "High"
        fix_description = (
            "Root cause устранён: выполнен ручной перезапуск upstream-пайплайна, "
            "после чего downstream-реквесты были перезапущены."
        )
    else:
        req_end_date = pd.Timestamp(bug_day).replace(hour=19, minute=40)
        fix_date = pd.Timestamp(bug_day).replace(hour=20, minute=30)
        priority = "Medium"
        fix_description = (
            "Downstream-реквест восстановлен после успешного завершения upstream-зависимости."
        )

    req_id = max_req_id

    new_loaders.append({
        "ID": max_loader_id,
        "REQ_ID": req_id,
        "REQ_TYPE": "Daily",
        "REQ_STATUS": "FAILED",
        "REQ_BEGIN_DATE": create_date,
        "REQ_END_DATE": req_end_date,
        "UPDATEDT": pd.Timestamp.now()
    })

    new_bugs.append({
        "ID": max_bug_id,
        "PID": ticker_pid.get(ticker),
        "REQ_ID": req_id,
        "PROJECT_NAME": ticker,
        "OID": ticker_oid.get(ticker),
        "OBJECT_NAME": ticker_object.get(ticker, f"STAGE_MOEX_{ticker}_RAW"),
        "SLA_FALL": 1,
        "SLA_FALL_DATE": sla_time,
        "ETA_FALL_DATE": sla_time + timedelta(minutes=20),
        "PRIORITY": priority,
        "CREATE_DATE": create_date,
        "FIX_DATE": fix_date,
        "BUG_DETAILS": (
            f"{reason} Target Object: "
            f"[{ticker_object.get(ticker, f'STAGE_MOEX_{ticker}_RAW')}]. "
            f"Каскадная роль: {role}."
        ),
        "FIX_DESCRIPTION": fix_description,
        "RESPONSIBLE": np.random.choice(["Терпунов А.А.", "Гайсин К.Д.", "Куликов Н.С."]),
        "UPDATEDT": pd.Timestamp.now()
    })

    max_bug_id += 1
    max_loader_id += 1
    max_req_id += 1

    return req_id


# ---------------------------------------------------------------------
# 4. Создание каскадных инцидентов и зависимостей
# ---------------------------------------------------------------------

for case in cascade_cases:
    bug_day = case["day"]

    root_req_id = get_or_create_incident(
        ticker=case["root"],
        bug_day=bug_day,
        reason=case["root_reason"],
        role="root"
    )

    for child in case["children"]:
        child_req_id = get_or_create_incident(
            ticker=child,
            bug_day=bug_day,
            reason=case["child_reason"],
            role="downstream"
        )

        new_dependencies.append({
            "I_REQ_ID": root_req_id,
            "UP_REQ_ID": child_req_id,
            "I_SCHEMA_NAME": case["schema"],
            "COMMENTS": (
                f"{case['domain_comment']}: "
                f"Root={case['root']} → Downstream={child}, "
                f"Дата={bug_day}. "
                f"Если I_REQ_ID={root_req_id} не завершился успешно, "
                f"UP_REQ_ID={child_req_id} мог упасть как следствие."
            )
        })

# ---------------------------------------------------------------------
# 5. Обновление датафреймов
# ---------------------------------------------------------------------

df_bugs_root = pd.concat([df_bugs_root, pd.DataFrame(new_bugs)], ignore_index=True)
df_loaders_root = pd.concat([df_loaders_root, pd.DataFrame(new_loaders)], ignore_index=True)
df_dependencies_root = pd.concat([df_dependencies_root, pd.DataFrame(new_dependencies)], ignore_index=True)

df_dependencies_root = df_dependencies_root.drop_duplicates(
    subset=["I_REQ_ID", "UP_REQ_ID"],
    keep="last"
)

df_bugs_root = df_bugs_root.drop(columns=["BUG_DAY"], errors="ignore")

# Возвращаем в основные переменные ноутбука
df_bugs = df_bugs_root.copy()
df_loaders = df_loaders_root.copy()
df_dependencies = df_dependencies_root.copy()

# ---------------------------------------------------------------------
# 6. Сохранение CSV
# ---------------------------------------------------------------------

df_bugs.to_csv("table_bugs.csv", index=False, encoding="utf-8-sig")
df_loaders.to_csv("table_loaders.csv", index=False, encoding="utf-8-sig")
df_dependencies.to_csv("table_dependencies.csv", index=False, encoding="utf-8-sig")

print("Создано новых bugs:", len(new_bugs))
print("Создано новых loaders:", len(new_loaders))
print("Создано новых dependencies:", len(new_dependencies))
print("Всего bugs:", len(df_bugs))
print("Всего loaders:", len(df_loaders))
print("Всего dependencies:", len(df_dependencies))

display(pd.DataFrame(new_dependencies))

Создано новых bugs: 0
Создано новых loaders: 0
Создано новых dependencies: 14
Всего bugs: 1127
Всего loaders: 6315
Всего dependencies: 54


,I_REQ_ID,UP_REQ_ID,I_SCHEMA_NAME,COMMENTS
0,56301,56302,STAGE_MOEX,Финансовая каскадная зависимость: Root=SBER → ...
1,56301,56303,STAGE_MOEX,Финансовая каскадная зависимость: Root=SBER → ...
2,56301,56304,STAGE_MOEX,Финансовая каскадная зависимость: Root=SBER → ...
3,56301,50672,STAGE_MOEX,Финансовая каскадная зависимость: Root=SBER → ...
4,56305,56306,STAGE_MOEX,Нефтегазовая каскадная зависимость: Root=GAZP ...
5,56305,56307,STAGE_MOEX,Нефтегазовая каскадная зависимость: Root=GAZP ...
6,56305,56308,STAGE_MOEX,Нефтегазовая каскадная зависимость: Root=GAZP ...
7,56305,56309,STAGE_MOEX,Нефтегазовая каскадная зависимость: Root=GAZP ...
8,56310,56311,STAGE_MOEX,ИТ-каскадная зависимость: Root=YNDX → Downstre...
9,56310,51406,STAGE_MOEX,ИТ-каскадная зависимость: Root=YNDX → Downstre...


In [25]:
import csv
filename = 'ticker_sectors.csv'

# Открываем файл на запись в кодировке utf-8
with open(filename, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f, delimiter=',')
    
    # Записываем заголовок таблицы
    writer.writerow(['ticker', 'sector'])
    
    # Построчно переносим тикеры и их сектора
    for ticker, sector in ticker_to_sector.items():
        writer.writerow([ticker, sector])

print(f"Файл '{filename}' успешно создан! Всего строк записано: {len(ticker_to_sector)}")

Файл 'ticker_sectors.csv' успешно создан! Всего строк записано: 315
